# Pump: support bearing — sensor vs camera

| | Sensor | Camera |
|---|---|---|
| Data | University of Ottawa vibration (bearing test rig) | XJTU event camera (lab pump) |
| Classes | healthy, inner race, outer race, ball fault | same four |
| Tested on | 24 recordings never used for training | 24 recordings never used for training |
| Code | `Models/pump.py sensor` | `Models/pump.py camera` |

The two branches use different lab rigs, so they are compared by fault class only. Lab data, not a mining pump; it classifies the present condition, not when to repair.

All of this machine's experiment code is in its `Models/` file; shared notebook set-up is in `notebook_helpers.py`. Run the cells from top to bottom with the **Capstone (.venv)** kernel.

In [1]:
from notebook_helpers import *

LOCAL = setup([
    ("Sensor: Ottawa vibration", "Raw_Data/uottawa_bearing", "*.mat", 48),
    ("Camera: XJTU-DV-Pump zips", "Raw_Data/xjtu_dv_pump", "*.zip", 2),
    ("Extra: original pump XLSX", "Raw_Data/pump_bearing", "*.xlsx", 4),
    ("Extra: UCI hydraulic", "Raw_Data/uci_hydraulic", "*.zip", 1),
    ("Extra: ESPset V3", "Raw_Data/espset_v3", "*.zip", 1),
])

,environment
git commit,080e7ac
python,3.11.9
kernel,project .venv
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2


,location,files,MB,local
data,,,,
Sensor: Ottawa vibration,Raw_Data/uottawa_bearing,48/48,631.5,True
Camera: XJTU-DV-Pump zips,Raw_Data/xjtu_dv_pump,2/2,16488.7,True
Extra: original pump XLSX,Raw_Data/pump_bearing,4/4,3.1,True
Extra: UCI hydraulic,Raw_Data/uci_hydraulic,1/1,76.6,True
Extra: ESPset V3,Raw_Data/espset_v3,1/1,0.3,True


All data is local: running fully offline.


## 1. Sensor: vibration
Train on speed profiles A and B, test on 24 separate C and D recordings (one answer per recording).

In [2]:
before = load("pump_bearing_independent.json")
if LOCAL["Sensor: Ottawa vibration"]:
    run_script("Models/pump.py", "sensor")
sensor = load("pump_bearing_independent.json")
check(before, sensor, [("random_forest_run_level",)], "Sensor")
show({"Sensor model (24 recordings)": sensor["random_forest_run_level"],
      "Always guess one class": sensor["dummy_run_level"]})

[Models/pump.py sensor]
  Saved Results\pump_bearing_independent.json and Figures\pump_bearing_independent_confusions.png
  Held-out run accuracy 0.9583; macro F1 0.9580
Sensor: matches the saved result


,accuracy,macro_precision,macro_recall,macro_f1
Sensor model (24 recordings),0.958,0.964,0.958,0.958
Always guess one class,0.250,0.062,0.250,0.100


## 2. Camera: event images with OpenCV
Eight 250 ms event images per recording, OpenCV HOG features. Train on 20 Hz, test on 30 Hz. Extra check: hide each camera set-up in turn.

In [3]:
before = load("pump_event_visual.json")
if LOCAL["Camera: XJTU-DV-Pump zips"]:
    use_local_pump_zips()
    run_script("Models/pump.py", "camera", "--refresh")
camera = load("pump_event_visual.json")
check(before, camera, [("random_forest_run_level",), ("held_out_camera_configuration_run_level",)], "Camera")
show({"Camera model (24 recordings)": camera["random_forest_run_level"],
      "Camera, unseen set-up (48)": camera["held_out_camera_configuration_run_level"],
      "Always guess one class": camera["dummy_run_level"]})

[Models/pump.py camera --refresh]
  30 Hz 24/24: outer 20,681,996 events
  Saved Results\pump_event_visual.json and Figures\pump_event_visual_confusion.png
  Run-level test macro F1: 1.0000
Camera: matches the saved result


,accuracy,macro_precision,macro_recall,macro_f1
Camera model (24 recordings),1.000,1.000,1.000,1.000
"Camera, unseen set-up (48)",0.854,0.868,0.854,0.855
Always guess one class,0.250,0.062,0.250,0.100


## 3. Sensor vs camera

In [4]:
run_script("Models/pump.py", "compare", tail=1)
show({"Sensor: vibration": sensor["random_forest_run_level"],
      "Camera: event images": camera["random_forest_run_level"],
      "Camera: unseen set-up": camera["held_out_camera_configuration_run_level"]})

[Models/pump.py compare]
  Saved Results\pump_sensor_visual_comparison.md and Figures\pump_sensor_visual_confusions.png


,accuracy,macro_precision,macro_recall,macro_f1
Sensor: vibration,0.958,0.964,0.958,0.958
Camera: event images,1.000,1.000,1.000,1.000
Camera: unseen set-up,0.854,0.868,0.854,0.855


![Pump sensor vs camera](../Figures/pump_sensor_visual_confusions.png)

**What this means:** both work well here (sensor 23/24 = 0.958, camera 24/24 = 1.000; guessing gets 0.250). The camera drops to about 0.85 on an unseen camera set-up. Different rigs, so this is not a winner contest. Full write-up: [`Results/pump_sensor_visual_comparison.md`](../Results/pump_sensor_visual_comparison.md).

## 4. Extra sensor-only pump checks (no camera branch)
UCI hydraulic and ESPset give slightly different numbers under scikit-learn 1.9.1 than the saved ones (a known version issue).

In [5]:
extras = [("Extra: original pump XLSX", "original-xlsx", "pump_bearing_exploratory.json", "random_forest"),
          ("Extra: UCI hydraulic", "uci-hydraulic", "uci_hydraulic_pump_leakage.json", "test_selected"),
          ("Extra: ESPset V3", "espset", "espset_v3_binary_fault.json", "test_selected")]
saved = {name: load(name) for _, _, name, _ in extras}
for key, command, name, part in extras:
    if LOCAL[key]:
        run_script("Models/pump.py", command, tail=1)
show({f"{key} (saved)": saved[name][part] for key, _, name, part in extras} |
     {f"{key} (this run)": load(name)[part] for key, _, name, part in extras})

[Models/pump.py original-xlsx]
  RF macro F1 1.0000; scientifically valid independent-run score: unavailable


[Models/pump.py uci-hydraulic]
  Wrote Results\uci_hydraulic_pump_leakage.json and Figures\uci_hydraulic_pump_leakage.png


[Models/pump.py espset]
  Wrote Results\espset_v3_binary_fault.json and Figures\espset_v3_binary_fault.png


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
Extra: original pump XLSX (saved),1.000,1.000,1.0,1.0,1.000
Extra: UCI hydraulic (saved),0.970,0.972,NaN,NaN,0.970
Extra: ESPset V3 (saved),0.939,0.907,NaN,NaN,0.918
Extra: original pump XLSX (this run),1.000,1.000,1.0,1.0,1.000
Extra: UCI hydraulic (this run),0.962,0.963,NaN,NaN,0.963
Extra: ESPset V3 (this run),0.937,0.904,NaN,NaN,0.914


**Data (CC BY 4.0):** University of Ottawa bearing vibration V2, DOI 10.17632/v43hmbwxpm.2; XJTU-DV, DOI 10.5281/zenodo.17447083.